# process captions collected from other sources

### FVQA
- coco 2014 val only
- ILSVRC 2012 test only

### AOKVQA
- coco 2017 train and val

In [1]:
import os
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit/data_raw")
import sys
from pathlib import Path
from tokens import openai_key, HF_TOKEN

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm import *
import argparse
args = argparse.Namespace(split="all", dataset_name="aokvqa")
config = configure_args(args, config_path=None)
ds = VQADataset(config)
aokvqa_df = ds.load_df()

args = argparse.Namespace(split="all", dataset_name="fvqa")
config = configure_args(args, config_path=None)
ds = VQADataset(config)
fvqa_df = ds.load_df()

import os
os.chdir(NOTEBOOK_ROOT/"data_raw")

Task evaluation metrics will be saved to results/te/ft/llava-1.5-7b-hf/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/llava-1.5-7b-hf/aokvqa
Predictions will be saved to results/pred/llava-1.5-7b-hf/aokvqa
Post-edit predictions will be saved to results/pred_postedit/ft/llava-1.5-7b-hf/aokvqa
Unified filename to save: mc_all.json
Task evaluation metrics will be saved to results/te/ft/llava-1.5-7b-hf/fvqa
Edit evaluation metrics will be saved to results/ee/ft/llava-1.5-7b-hf/fvqa
Predictions will be saved to results/pred/llava-1.5-7b-hf/fvqa
Post-edit predictions will be saved to results/pred_postedit/ft/llava-1.5-7b-hf/fvqa
Unified filename to save: mc_all.json


## coco 2017

In [ ]:
# # Create directory structure
# !mkdir -p coco/coco2017

# # Download annotation zip (includes captions) to coco/coco2017/
# !wget -P coco/coco2017 http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# # Unzip just the caption files
# !cd coco/coco2017 && unzip annotations_trainval2017.zip annotations/captions_train2017.json annotations/captions_val2017.json

# # Move files from annotations/ subdirectory to coco/coco2017/
# !mv coco/coco2017/annotations/captions_train2017.json coco/coco2017/
# !mv coco/coco2017/annotations/captions_val2017.json coco/coco2017/

# # Clean up: remove annotations subdirectory and zip file
# !rm -rf coco/coco2017/annotations coco/coco2017/annotations_trainval2017.zip


## coco 2014

In [ ]:
# # Create directory structure
# !mkdir -p coco/coco2014

# # Download annotation zip (includes captions) to coco/coco2014/
# !wget -P coco/coco2014 http://images.cocodataset.org/annotations/annotations_trainval2014.zip

# # Unzip just the caption files
# !cd coco/coco2014 && unzip annotations_trainval2014.zip annotations/captions_train2014.json annotations/captions_val2014.json

# # Move files from annotations/ subdirectory to coco/coco2014/
# !mv coco/coco2014/annotations/captions_train2014.json coco/coco2014/
# !mv coco/coco2014/annotations/captions_val2014.json coco/coco2014/

# # Clean up: remove annotations subdirectory and zip file
# !rm -rf coco/coco2014/annotations coco/coco2014/annotations_trainval2014.zip

In [4]:
import json
import pandas as pd

def load_coco_captions(year=2017, split='val', base_path='./coco'):
    # Construct file path
    file_path = f"{base_path}/coco{year}/captions_{split}{year}.json"
    # Load JSON file
    with open(file_path) as f:
        caps = json.load(f)
    # Collect all annotations into a list of dictionaries
    data = []
    for annotation in caps["annotations"]:
        data.append({
            "image_info_id": split+"_"+str(annotation['image_id']),
            "caption": annotation['caption']
        })
    # Create DataFrame from all collected data
    cap_df = pd.DataFrame(data)
    # Keep only the first caption per image
    cap_df = cap_df.drop_duplicates(subset=['image_info_id'], keep='first') 
    return cap_df

# Load all the datasets you need
cap_df2017_train = load_coco_captions(year=2017, split='train')
cap_df2017_val = load_coco_captions(year=2017, split='val')
cap_df2017 = pd.concat([cap_df2017_train, cap_df2017_val])
cap_df2014_train = load_coco_captions(year=2014, split='train')
cap_df2014_val = load_coco_captions(year=2014, split='val')
cap_df2014 = pd.concat([cap_df2014_train, cap_df2014_val])

# Print summary
print(f"2017 Train: {len(cap_df2017_train)} captions")
print(f"2017 Val: {len(cap_df2017_val)} captions")
print(f"2014 Train: {len(cap_df2014_train)} captions")
print(f"2014 Val: {len(cap_df2014_val)} captions")

aokvqa_df = aokvqa_df.merge(cap_df2017, on="image_info_id", how="left")
fvqa_df = fvqa_df.merge(cap_df2014, on="image_info_id", how="left")
# Deduplicate: keep only LAST cot entry per image_path (avoid explosion from multiple questions per image)
aokvqa_df_caption = aokvqa_df[['image_info_source', 'image_info_id', 'image_path', 'cot', 'caption']].drop_duplicates(subset=['image_path'], keep='last')
fvqa_df_caption = fvqa_df[['image_info_source', 'image_info_id', 'image_path', 'cot', 'caption']].drop_duplicates(subset=['image_path'], keep='last')

PARQUET_DIR = Path("../data/captions/parquet/")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
fvqa_df_caption.to_parquet(PARQUET_DIR / "fvqa.parquet", index=False)
aokvqa_df_caption.to_parquet(PARQUET_DIR / "aokvqa.parquet", index=False)

2017 Train: 118287 captions
2017 Val: 5000 captions
2014 Train: 82783 captions
2014 Val: 40504 captions


In [ ]:
# Count unique image_path for rows with and without captions
has_caption = fvqa_df['caption'].notna()
print(f"Images WITH caption: {fvqa_df[has_caption]['image_path'].nunique()} unique")
print(f"Images WITHOUT caption: {fvqa_df[~has_caption]['image_path'].nunique()} unique")
print(f"Total unique images: {fvqa_df['image_path'].nunique()}")

# Count unique image_path for rows with and without captions
has_caption = aokvqa_df['caption'].notna()
print(f"Images WITH caption: {aokvqa_df[has_caption]['image_path'].nunique()} unique")
print(f"Images WITHOUT caption: {aokvqa_df[~has_caption]['image_path'].nunique()} unique")
print(f"Total unique images: {aokvqa_df['image_path'].nunique()}")


Images WITH caption: 1332 unique
Images WITHOUT caption: 858 unique
Total unique images: 2190
Images WITH caption: 17656 unique
Images WITHOUT caption: 0 unique
Total unique images: 17656


In [6]:
from huggingface_hub import HfApi, create_repo, upload_folder
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi(token=HF_TOKEN)
repo_id = "JJoy333/RationaleVQA"
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)


upload_folder(
    folder_path=str(PARQUET_DIR),
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="/i_gen"
)


No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/JJoy333/RationaleVQA/commit/1686131bca57be5f1679d47e5a728bbab7c4f95f', commit_message='Upload folder using huggingface_hub', commit_description='', oid='1686131bca57be5f1679d47e5a728bbab7c4f95f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JJoy333/RationaleVQA', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JJoy333/RationaleVQA'), pr_revision=None, pr_num=None)

## ILSVRC 2012

- only train split is available from CLIP
- have to create caption from image using QWEN3

In [ ]:
# !mkdir -p coco/ilsvrc2012
# !wget -O coco/ilsvrc2012/imagenet_captions.zip https://raw.githubusercontent.com/mlfoundations/imagenet-captions/main/imagenet_captions.zip
# !cd coco/ilsvrc2012 && unzip imagenet_captions.zip && rm imagenet_captions.zip

# import json

# with open("./coco/ilsvrc2012/imagenet_captions.json") as f:
#     data = json.load(f)

# def build_caption(entry):
#     title = (entry.get("title") or "").strip()
#     desc = (entry.get("description") or "").strip()
#     tags = entry.get("tags") or []

#     parts = []
#     if title:
#         parts.append(title)
#     if desc:
#         parts.append(desc)
#     if tags:
#         parts.append(" ".join(tags))

#     return " ".join(parts)

# def build_image_info_id(entry):
#     return entry["filename"].split(".")[0].split("_")[-1]

# def build_caption_df(data):
#     df = []
#     for entry in data:
#         image_info_id = build_image_info_id(entry)
#         caption = build_caption(entry)
#         df_e = {
#             'image_info_id': image_info_id,
#             'caption': caption
#         }
#         df.append(df_e)
#     # keep only the first caption per image
#     df = pd.DataFrame(df)
#     df = df.drop_duplicates(subset=['image_info_id'], keep='first')
#     return df




In [8]:
# Load captions from HuggingFace
from huggingface_hub import snapshot_download
import pandas as pd
import os
# Download caption parquet files from HF
repo_id = "JJoy333/RationaleVQA"
local_root = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["i_gen/*.parquet"],
)
# Load caption dataframes
fvqa_df_caption = pd.read_parquet(os.path.join(local_root, "i_gen", "fvqa.parquet"))
print(f"Loaded FVQA captions: {len(fvqa_df_caption)} rows")
print(f"\nFVQA caption columns: {fvqa_df_caption.columns.tolist()}")



from PIL import Image
import torch
from tqdm import tqdm
import os
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")
# Initialize Qwen3 model for caption generation
args_caption = argparse.Namespace(
    model_name="qwen3",
    dataset_name="fvqa",  # dummy, just for config
    split="all"
)
config_caption = configure_args(args_caption, config_path=None)
config_caption.device = "cuda" if torch.cuda.is_available() else "cpu"
caption_model = VQAModel(config_caption).to(config_caption.device)
caption_model.eval()

Loaded FVQA captions: 2190 rows

FVQA caption columns: ['image_info_source', 'image_info_id', 'image_path', 'cot', 'caption']
Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-8B-Instruct/fvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-8B-Instruct/fvqa
Predictions will be saved to results/pred/Qwen3-VL-8B-Instruct/fvqa
Post-edit predictions will be saved to results/pred_postedit/ft/Qwen3-VL-8B-Instruct/fvqa
Unified filename to save: mc_all.json


VQAModel(
  (model): Qwen3VLForConditionalGeneration(
    (model): Qwen3VLModel(
      (visual): Qwen3VLVisionModel(
        (patch_embed): Qwen3VLVisionPatchEmbed(
          (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
        )
        (pos_embed): Embedding(2304, 1152)
        (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
        (blocks): ModuleList(
          (0-26): 27 x Qwen3VLVisionBlock(
            (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (attn): Qwen3VLVisionAttention(
              (qkv): Linear(in_features=1152, out_features=3456, bias=True)
              (proj): Linear(in_features=1152, out_features=1152, bias=True)
            )
            (mlp): Qwen3VLVisionMLP(
              (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
              (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
       

In [9]:
# Caption generation prompt
CAPTION_PROMPT = "Describe this image in a short sentence."

# Function to generate captions for a batch of images
def generate_captions_batch(image_paths, model, prompt, batch_size=8, max_new_tokens=100):
    """Generate captions for a batch of images."""
    captions = []
    
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Generating captions"):
        batch_paths = image_paths[i:i+batch_size]
        
        # Load images
        images = []
        valid_indices = []
        for idx, img_path in enumerate(batch_paths):
            try:
                img = Image.open(img_path).convert("RGB")
                # Resize if needed (similar to VQADataset)
                w, h = img.size
                max_side = 800
                m = max(w, h)
                if m > max_side:
                    s = max_side / m
                    img = img.resize((int(w * s), int(h * s)), Image.BICUBIC)
                images.append(img)
                valid_indices.append(i + idx)
            except Exception as e:
                print(f"Error loading image {img_path}: {e}")
                continue
        
        if not images:
            continue
            
        # Generate captions
        try:
            prompts = [prompt] * len(images)
            batch_captions = model.generate(
                images, 
                prompts, 
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True
            )
            
            # Store captions (with None for failed images)
            for valid_idx, caption in zip(valid_indices, batch_captions):
                captions.append((valid_idx, caption))
        except Exception as e:
            print(f"Error generating captions for batch {i}: {e}")
            # Add None for this batch
            for valid_idx in valid_indices:
                captions.append((valid_idx, None))
    
    return captions

# Get ImageNet subset images
fvqa_df_imagenet = fvqa_df_caption[fvqa_df_caption['image_info_source'] == "ILSVRC"]
fvqa_df_imagenet = fvqa_df_imagenet.reset_index(drop=True)
print(f"Found {len(fvqa_df_imagenet)} ImageNet images to caption")

# Generate captions
image_paths = fvqa_df_imagenet['image_path'].tolist()
# image_paths = image_paths[:20]
caption_results = generate_captions_batch(
    image_paths, 
    caption_model, 
    CAPTION_PROMPT,
    batch_size=20,  # Adjust based on GPU memory
    max_new_tokens=30
)

# Create a mapping from index to caption
caption_dict = {idx: caption for idx, caption in caption_results}



Found 858 ImageNet images to caption


Generating captions: 100%|█████████████████████████████████████████████| 43/43 [01:20<00:00,  1.87s/it]


In [10]:
fvqa_df_imagenet = fvqa_df_caption[fvqa_df_caption['image_info_source'] == "ILSVRC"]
fvqa_df_imagenet = fvqa_df_imagenet.reset_index(drop=True)
fvqa_df_imagenet['caption'] = fvqa_df_imagenet.index.map(caption_dict)

# # Check how many captions were generated
# print(f"Generated {fvqa_df_imagenet['caption'].notna().sum()} captions out of {len(fvqa_df_imagenet)} images")

# # Show some examples
# print("\nSample generated captions:")
# for idx, row in fvqa_df_imagenet.iterrows():
#     if pd.notna(row['caption']):
#         print(f"Image {row['image_info_id']}: {row['caption']}")

fvqa_df_coco = fvqa_df_caption[fvqa_df_caption['image_info_source'] != "ILSVRC"]
fvqa_df_caption = pd.concat([fvqa_df_coco, fvqa_df_imagenet], axis=0, ignore_index=True)
# Deduplicate: keep only LAST cot entry per image_path
fvqa_df_caption = fvqa_df_caption.drop_duplicates(subset=['image_path'], keep='last')
PARQUET_DIR = Path("./data/captions/parquet/")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
fvqa_df_caption.to_parquet(PARQUET_DIR / "fvqa.parquet", index=False)


In [11]:
import os
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit/data_raw")
import sys
from pathlib import Path
from tokens import openai_key, HF_TOKEN

from huggingface_hub import HfApi, create_repo, upload_folder
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit/")
PARQUET_DIR = Path("./data/captions/parquet/")
api = HfApi(token=HF_TOKEN)
repo_id = "JJoy333/RationaleVQA"
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)


upload_folder(
    folder_path=str(PARQUET_DIR),
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="i_gen"
)


CommitInfo(commit_url='https://huggingface.co/datasets/JJoy333/RationaleVQA/commit/0913f29c4313eb7a2c35f75a8d68167050c81e13', commit_message='Upload folder using huggingface_hub', commit_description='', oid='0913f29c4313eb7a2c35f75a8d68167050c81e13', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JJoy333/RationaleVQA', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JJoy333/RationaleVQA'), pr_revision=None, pr_num=None)

In [12]:
# from huggingface_hub import HfApi
# from tokens import HF_TOKEN  # or however you get your token

# api = HfApi(token=HF_TOKEN)
# repo_id = "JJoy333/RationaleVQA"  # your repo
# path_in_repo = "i_gen/caption"  # the path you want to delete

# # Delete a single file or directory
# api.delete_file(
#     path_in_repo=path_in_repo,
#     repo_id=repo_id,
#     repo_type="dataset"  # or "model" or None for default
# )

# load it back

In [13]:
# Load caption dataframes for both datasets
from huggingface_hub import snapshot_download
import pandas as pd
import os

repo_id = "JJoy333/RationaleVQA"
local_root = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["i_gen/*.parquet"],
)

fvqa_df_caption = pd.read_parquet(os.path.join(local_root, "i_gen", "fvqa.parquet"))
aokvqa_df_caption = pd.read_parquet(os.path.join(local_root, "i_gen", "aokvqa.parquet"))

print(f"FVQA captions: {len(fvqa_df_caption)} rows")
print(f"AOKVQA captions: {len(aokvqa_df_caption)} rows")


FVQA captions: 2190 rows
AOKVQA captions: 17656 rows


In [14]:
aokvqa_df_caption.caption[0]

'Man dressed in jeans standing in the street holding luggage.'